In [1]:
import seaborn as sns

tips = sns.load_dataset('tips')
df = tips.copy()
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [2]:
mediana = df['tip'].median()
print(mediana)

2.9


In [3]:
df['generosa'] = df['tip'] >= mediana
df.head()

,total_bill,tip,sex,smoker,day,time,size,generosa
0,16.99,1.01,Female,No,Sun,Dinner,2,False
1,10.34,1.66,Male,No,Sun,Dinner,3,False
2,21.01,3.50,Male,No,Sun,Dinner,3,True
3,23.68,3.31,Male,No,Sun,Dinner,2,True
4,24.59,3.61,Female,No,Sun,Dinner,4,True


In [4]:
df['generosa'].value_counts()

generosa
False    122
True     122
Name: count, dtype: int64

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
 7   generosa    244 non-null    bool    
dtypes: bool(1), category(4), float64(2), int64(1)
memory usage: 7.6 KB


In [6]:
#Transformações
#sex, smoker, time
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.get_dummies(df, columns=['sex', 'smoker', 'time'], dtype=int, drop_first=True)
ordinal = OrdinalEncoder(categories=[['Thur','Fri','Sat','Sun']])
df['day_encoder'] = ordinal.fit_transform(df[['day']])
df.drop('day', axis=1, inplace=True)
df.head()


,total_bill,tip,size,generosa,sex_Female,smoker_No,time_Dinner,day_encoder
0,16.99,1.01,2,False,1,1,1,3.0
1,10.34,1.66,3,False,0,1,1,3.0
2,21.01,3.50,3,True,0,1,1,3.0
3,23.68,3.31,2,True,0,1,1,3.0
4,24.59,3.61,4,True,1,1,1,3.0


In [7]:
#Separar atributos preditivos (X) e o alvo (y)
X = df.drop(['generosa','tip'],axis=1).copy()
y = df['generosa']
X.head()
y.head()

0    False
1    False
2     True
3     True
4     True
Name: generosa, dtype: bool

In [8]:
pd.concat([X.head(), y.head()])

,total_bill,size,sex_Female,smoker_No,time_Dinner,day_encoder,generosa
0,16.99,2.0,1.0,1.0,1.0,3.0,NaN
1,10.34,3.0,0.0,1.0,1.0,3.0,NaN
2,21.01,3.0,0.0,1.0,1.0,3.0,NaN
3,23.68,2.0,0.0,1.0,1.0,3.0,NaN
4,24.59,4.0,1.0,1.0,1.0,3.0,NaN
0,NaN,NaN,NaN,NaN,NaN,NaN,False
1,NaN,NaN,NaN,NaN,NaN,NaN,False
2,NaN,NaN,NaN,NaN,NaN,NaN,True
3,NaN,NaN,NaN,NaN,NaN,NaN,True
4,NaN,NaN,NaN,NaN,NaN,NaN,True


In [9]:
#Classificando a gorjeta

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    random_state=42,
                                                   stratify=y)

print(len(X_train))
print(len(y_train))
print(y_train.value_counts(normalize=True))
print(len(X_test))
print(len(y_test))
print(y_test.value_counts(normalize=True))

170
170
generosa
True     0.5
False    0.5
Name: proportion, dtype: float64
74
74
generosa
True     0.5
False    0.5
Name: proportion, dtype: float64


In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train) #treinamento
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[25 12]
 [ 9 28]]


In [11]:
cm = confusion_matrix(y_test, y_pred, labels=[1,0])
print(cm)

[[28  9]
 [12 25]]


In [12]:
#Acurácia
print((cm[0,0]+cm[1,1])/74)

0.7162162162162162


In [19]:
VP = cm[0,0]
VN = cm[1,1]
FP = cm[1,0]
FN = cm[0,1]
total = VP+VN+FP+FN
acuracia = (VP+VN) / total
precisao_positiva = VP / (VP+FP)
recall_positiva = VP / (VP+FN)
precisao_negativa = VN / (VN+FN)
recall_negativa = VN / (VN+FP)
f1_positiva = 2 * (precisao_positiva*recall_positiva) / (precisao_positiva+recall_positiva)
f1_negativa = 2 * (precisao_negativa*recall_negativa) / (precisao_negativa+recall_negativa)
print(f'Acurácia: {acuracia*100:.2f}%')
print(f'Precisão da classe Positiva (True) : {precisao_positiva:.2f}')
print(f'Recall da classe Positiva (True) : {recall_positiva:.2f}')
print(f'F1 da classe Positiva (True) : {f1_positiva:.2f}')
print(f'Precisão da classe Negativa (False) : {precisao_negativa:.2f}')
print(f'Recall da classe Negativa (False) : {recall_negativa:.2f}')
print(f'F1 da classe Negativa (False) : {f1_negativa:.2f}')

Acurácia: 71.62%
Precisão da classe Positiva (True) : 0.70
Recall da classe Positiva (True) : 0.76
F1 da classe Positiva (True) : 0.73
Precisão da classe Negativa (False) : 0.74
Recall da classe Negativa (False) : 0.68
F1 da classe Negativa (False) : 0.70


In [20]:
from sklearn.metrics import classification_report

relatorio = classification_report(y_test, y_pred)
print(relatorio)


              precision    recall  f1-score   support

       False       0.74      0.68      0.70        37
        True       0.70      0.76      0.73        37

    accuracy                           0.72        74
   macro avg       0.72      0.72      0.72        74
weighted avg       0.72      0.72      0.72        74



In [23]:
from sklearn.model_selection import StratifiedKFold, cross_validate
model = RandomForestClassifier(random_state=42, n_estimators=100) # Modelo
cv_est = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
result_cv = cross_validate( # Executar cross_validate com múltiplas métricas
model, X, y,
cv=cv_est,
scoring=['accuracy', 'precision', 'recall', 'f1'],
return_train_score=False
) # Exibir resultados em formato de tabela
df_result = pd.DataFrame(result_cv).drop('fit_time', axis=1).drop('score_time', axis=1)
df_result.columns = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
df_result.index = [f'Fold {i+1}' for i in range(5)]
df_result.loc['Média'] = df_result.mean()
df_result.loc['Desvio Padrão'] = df_result.iloc[:5].std()
print('RESULTADOS DA VALIDAÇÃO CRUZADA')
print('═' * 60)
print(df_result.to_string(float_format=lambda x: f'{x:.4f}'))

RESULTADOS DA VALIDAÇÃO CRUZADA
════════════════════════════════════════════════════════════
               Acurácia  Precisão  Recall  F1-Score
Fold 1           0.6939    0.6957  0.6667    0.6809
Fold 2           0.7959    0.8182  0.7500    0.7826
Fold 3           0.6327    0.6522  0.6000    0.6250
Fold 4           0.7347    0.7000  0.8400    0.7636
Fold 5           0.6458    0.6400  0.6667    0.6531
Média            0.7006    0.7012  0.7047    0.7010
Desvio Padrão    0.0669    0.0705  0.0925    0.0690
